# GroupBy: lo que probablemente no conoces

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/14_pandas_transformaciones/code/03_groupby.ipynb)

---

Este notebook **no es una intro a groupby** — ya viste eso en cursos anteriores.

El objetivo es cubrir las partes que la mayoría no conoce y que son las más útiles en la práctica:

- **Named aggregations** — resultado limpio sin MultiIndex raro
- **`transform()`** — la operación que más se subestima
- **`filter()`** — filtrar grupos enteros, no filas individuales
- **`apply()`** con groupby — el martillo cuando todo lo demás falla
- Patrones comunes en ciencia de datos

Si ya dominas todo esto, usa el notebook como referencia rápida.

In [ ]:
%pip install pandas==2.2.3 numpy==1.26.4

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
df = pd.DataFrame({
    "empleado_id": range(1, 21),
    "nombre": [f"Emp_{i}" for i in range(1, 21)],
    "departamento": np.random.choice(["Ventas", "Tech", "RRHH", "Finanzas"], 20),
    "ciudad": np.random.choice(["CDMX", "MTY", "GDL"], 20),
    "salario": np.random.randint(25000, 90000, 20),
    "años_experiencia": np.random.randint(1, 15, 20),
    "activo": np.random.choice([True, False], 20, p=[0.8, 0.2]),
})

print(f"Shape: {df.shape}")
df.head(10)

## Sección 1: Breve recap — `agg()` ya lo conoces

Repaso rápido para alinear el vocabulario antes de ver lo nuevo.

In [ ]:
# Forma básica: una sola estadística
df.groupby("departamento")["salario"].mean()

In [ ]:
# Varias estadísticas a la vez
df.groupby("departamento")["salario"].agg(["mean", "std", "count"])

In [ ]:
# Agrupación por múltiples columnas
df.groupby(["departamento", "ciudad"])["salario"].mean()

**Punto clave:** `.agg()` *colapsa* los datos — el resultado tiene **una fila por grupo**, no una por empleado.

> 💡 **Prueba esto:** ¿Cuántas filas tiene cada resultado arriba vs el DataFrame original (20 filas)?

## Sección 2: Named aggregations — el resultado limpio

Cuando agrupas por múltiples columnas con el diccionario clásico, pandas genera columnas jerárquicas (MultiIndex) que son difíciles de manejar.

In [ ]:
# Forma antigua — columnas jerárquicas raras
resultado_feo = df.groupby("departamento").agg(
    {"salario": ["mean", "max"], "años_experiencia": "mean"}
)
print("Columnas:", resultado_feo.columns.tolist())
resultado_feo

In [ ]:
# Named aggregations (pandas >= 0.25) — columnas con nombres que tú eliges
resultado = df.groupby("departamento").agg(
    salario_promedio=pd.NamedAgg(column="salario", aggfunc="mean"),
    salario_max=pd.NamedAgg(column="salario", aggfunc="max"),
    experiencia_promedio=pd.NamedAgg(column="años_experiencia", aggfunc="mean"),
    n_empleados=pd.NamedAgg(column="empleado_id", aggfunc="count"),
)

print("Columnas:", resultado.columns.tolist())
resultado

Sin MultiIndex, sin columnas raras. El resultado es un DataFrame normal que puedes usar directamente.

También puedes usar lambdas como `aggfunc`:

> 💡 **Prueba esto:** Agrega una columna `porcentaje_activos` al resultado usando `aggfunc=lambda x: x.mean()` sobre la columna `activo` (que es bool, así que `.mean()` da la proporción de `True`).

## Sección 3: `transform()` — el que no conocen

Esta es la operación que más se subestima y más se usa en feature engineering.

**La diferencia fundamental con `agg()`:**
- `agg()` → colapsa a un valor por grupo (el resultado tiene menos filas)
- `transform()` → devuelve una Serie con la **misma longitud** que el DataFrame original

In [ ]:
# agg: colapsa a un valor por grupo
medias_agg = df.groupby("departamento")["salario"].agg("mean")
print(f"agg shape: {medias_agg.shape}")  # (4,) — solo 4 departamentos
print(medias_agg)
print()

In [ ]:
# transform: mantiene todos los registros originales
medias_transform = df.groupby("departamento")["salario"].transform("mean")
print(f"transform shape: {medias_transform.shape}")  # (20,) — todos los empleados
print(medias_transform.head(10))

In [ ]:
# Esto lo hace perfecto para agregar columnas nuevas al DataFrame original
df["salario_medio_depto"] = df.groupby("departamento")["salario"].transform("mean")

# Cada empleado ahora tiene el salario promedio de SU departamento
df[["nombre", "departamento", "salario", "salario_medio_depto"]].head(10)

**Por qué importa en ML:** Necesitas features a nivel fila, no a nivel grupo. `transform()` te permite crear features como "¿cuánto gana este empleado comparado con su departamento?" sin perder las filas originales.

In [ ]:
# Normalización dentro de grupo (z-score por departamento)
df["salario_norm_depto"] = (
    df["salario"] - df.groupby("departamento")["salario"].transform("mean")
) / df.groupby("departamento")["salario"].transform("std")

# Rank dentro de departamento (1 = el que menos gana en su depto)
df["salario_rank_depto"] = df.groupby("departamento")["salario"].transform("rank")

# Booleano: ¿gana más que la media de su departamento?
df["es_mayor_media"] = df["salario"] > df.groupby("departamento")["salario"].transform("mean")

df[["nombre", "departamento", "salario", "salario_norm_depto", "salario_rank_depto", "es_mayor_media"]].head(10)

> 💡 **Prueba esto:** ¿Puedes crear una columna `años_experiencia_pct` que sea la experiencia del empleado dividida entre la experiencia promedio de su departamento? (valores > 1 = más experiencia que el promedio del depto)

## Sección 4: `filter()` — filtrar grupos enteros

`filter()` no filtra filas individuales — filtra **grupos completos**.

La función recibe un grupo (DataFrame) y debe devolver `True` (conservar todo el grupo) o `False` (descartar todo el grupo).

In [ ]:
# Mantener solo departamentos con salario promedio > 50,000
df_filtrado = df.groupby("departamento").filter(
    lambda grupo: grupo["salario"].mean() > 50000
)

print(f"Filas originales:          {len(df)}")
print(f"Filas después de filter:   {len(df_filtrado)}")
print(f"Departamentos restantes:   {df_filtrado['departamento'].unique()}")

In [ ]:
# Diferencia importante: filter() vs query()

# query filtra FILA POR FILA — un empleado puede quedar aunque su depto tenga media baja
df_query = df.query("salario > 50000")
print(f"query (fila por fila):     {len(df_query)} filas")

# filter filtra GRUPOS ENTEROS — si el depto no cumple, pierdes a TODOS sus empleados
df_filter = df.groupby("departamento").filter(lambda g: g["salario"].mean() > 50000)
print(f"filter (grupo completo):   {len(df_filter)} filas")
print()
print("query conserva empleados con salario > 50k aunque su depto no pase el corte.")
print("filter elimina a TODOS los empleados de deptos que no cumplen la condición.")

> 💡 **Prueba esto:** Filtra para conservar solo los departamentos que tienen **más de 5 empleados activos** (`activo == True`).

## Sección 5: `apply()` con groupby — el más flexible (y el más lento)

Cuando `agg`, `transform` y `filter` no alcanzan, existe `apply`.

La función recibe un grupo (DataFrame) y puede devolver lo que quieras: un escalar, una Serie, otro DataFrame.

In [ ]:
# Caso que no se puede hacer fácil con agg: necesito columnas múltiples en la salida
def top_empleados(grupo, n=2):
    """Devuelve los n empleados con mayor salario en el grupo."""
    return grupo.nlargest(n, "salario")[["nombre", "salario"]]

# include_groups=False evita un DeprecationWarning en pandas >= 2.2
df.groupby("departamento").apply(top_empleados, n=2, include_groups=False)

**Advertencia:** `apply` es significativamente más lento que `agg`, `transform` y `filter` porque no puede vectorizarse igual. Úsalo solo cuando las otras opciones no funcionan.

> 💡 **Prueba esto:** ¿Puedes obtener el mismo resultado (top 2 por departamento) sin `groupby.apply`? Pista: `sort_values` + `groupby` + `.head(2)`.

In [ ]:
# Alternativa sin apply — generalmente más rápida
(
    df.sort_values("salario", ascending=False)
    .groupby("departamento")[["nombre", "salario"]]
    .head(2)
    .sort_values(["departamento", "salario"], ascending=[True, False])
)

## Sección 6: Patrones comunes en ciencia de datos

Estos son los patrones que vas a usar constantemente. Memorízalos o guarda este notebook como referencia.

In [ ]:
# Conteo por combinación de grupos
df.groupby(["departamento", "ciudad"]).size().reset_index(name="count")

In [ ]:
# Proporción del salario de cada empleado dentro de su departamento
total_por_depto = df.groupby("departamento")["salario"].transform("sum")
df["prop_salario"] = df["salario"] / total_por_depto

# Verificación: la proporción dentro de cada depto debe sumar 1
df.groupby("departamento")["prop_salario"].sum()

In [ ]:
# Primer/último registro por grupo después de ordenar
# Útil para: primer evento por usuario, registro más reciente, etc.
df.sort_values("salario").groupby("departamento").first()[["nombre", "salario", "años_experiencia"]]

In [ ]:
# Suma acumulada dentro de grupo
# Útil para: acumulado de ventas por vendedor, acumulado por mes, etc.
df_sorted = df.sort_values(["departamento", "salario"]).copy()
df_sorted["salario_acumulado_depto"] = (
    df_sorted.groupby("departamento")["salario"].transform("cumsum")
)
df_sorted[["departamento", "nombre", "salario", "salario_acumulado_depto"]].head(12)

## Sección 7: Ejercicio

Usa el DataFrame `df` con el que trabajamos todo el notebook. Los ejercicios van en orden de dificultad.

**Ejercicio 1 — Named aggregations**

Crea un resumen por departamento con las siguientes columnas:
- `salario_promedio` — promedio de salario
- `salario_max` — salario máximo
- `n_empleados` — número de empleados
- `pct_activos` — porcentaje de empleados activos (entre 0 y 1)

El resultado debe ser un DataFrame limpio sin MultiIndex.

In [ ]:
# Tu solución aquí


**Ejercicio 2 — transform()**

Agrega una columna `salario_percentil_depto` que indique el percentile rank del salario de cada empleado **dentro de su departamento** (entre 0 y 1).

Pista: `transform` acepta funciones personalizadas. Busca `pd.Series.rank` con `pct=True`.

In [ ]:
# Tu solución aquí


**Ejercicio 3 — filter()**

Filtra el DataFrame para conservar solo los departamentos donde **todos** los empleados están activos (`activo == True`).

¿Cuántos departamentos quedan? ¿Cuántas filas?

In [ ]:
# Tu solución aquí


**Ejercicio 4 — transform() avanzado**

Crea una columna `nivel` con dos categorías:
- `"senior"` si el salario del empleado es mayor que el percentil 75 de su departamento
- `"junior"` en cualquier otro caso

Pista: Primero calcula el p75 por departamento con `transform`, luego compara.

In [ ]:
# Tu solución aquí
